# Partie A : PatchCore sur Google Colab

1. Dans Google Drive, créez un dossier `projet_cv/` et mettez-y :
   - le dossier `partie_A_detection/` (dézippé),
   - `bottle.tar.xz` et `cable.tar.xz`.
2. Menu **Exécution > Modifier le type d'exécution > GPU (T4)**.
3. Exécutez les cellules dans l'ordre.

Le travail se fait sur le disque local de Colab, plus rapide que Drive. Les modèles et les résultats sont recopiés dans Drive à la fin.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Pas de GPU : activez-le dans Exécution > Modifier le type d exécution'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/projet_cv'   # à adapter si besoin
!ls "$DRIVE_DIR"

In [ ]:
# Copie locale du projet et des archives
!rm -rf /content/partie_A_detection
!cp -r "$DRIVE_DIR/partie_A_detection" /content/
!cp "$DRIVE_DIR"/bottle.tar.xz "$DRIVE_DIR"/cable.tar.xz /content/partie_A_detection/
%cd /content/partie_A_detection
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip show anomalib | head -2

### (Optionnel) Test rapide sur un faux dataset

In [ ]:
!bash tests/smoke_test.sh 2>&1 | tail -5

### Pipeline complet : données, entraînement, évaluation, analyse des erreurs

In [ ]:
!bash scripts/run_all.sh .

### Résultats

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/dataset_stats.md').read()))
display(Markdown(open('results/results_table.md').read()))

In [ ]:
for cat in ['bottle', 'cable']:
    for f in ['roc', 'score_hist', 'per_defect', 'best_detections', 'errors_fp_fn']:
        p = f'results/{cat}/figures/{f}.png'
        import os
        if os.path.exists(p):
            print(cat, f); display(Image(p, width=650))

### Test de `predict()` (la fonction utilisée par B)

In [ ]:
import glob
from detection.predict import predict, to_json
img = sorted(glob.glob('data/MVTecAD/cable/test/cut_outer_insulation/*.png'))[0]
res = predict(img, 'cable')
print(to_json(res))
!python -m detection.predict --category cable --image "$img" --out /content/overlay.png
display(Image('/content/overlay.png', width=400))

### Sauvegarde dans Drive (modèles + résultats)

In [ ]:
!mkdir -p "$DRIVE_DIR/partie_A_sorties"
!cp -r models results "$DRIVE_DIR/partie_A_sorties/"
!ls -la "$DRIVE_DIR/partie_A_sorties/models"

### (Bonus) Étude d'ablation

In [ ]:
# Plusieurs entraînements : compter un peu de temps
!python -m detection.ablation --category bottle
!cp -r results "$DRIVE_DIR/partie_A_sorties/"